In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("synthetic_real_time_fraud_transactions.csv")

In [3]:
print("total rows and columns :")
print(df.shape)

total rows and columns :
(100010, 14)


In [4]:
print("print all column names and convert them into python list")
print(df.columns.tolist())

print all column names and convert them into python list
['transaction_id', 'timestamp', 'card_id', 'merchant_id', 'merchant_category', 'amount', 'channel', 'card_present', 'country', 'international', 'amount_vs_card_avg', 'minutes_since_prev_txn', 'card_txns_previous_10m', 'is_fraud']


In [5]:
print("first 5 rows")
display(df.head())

first 5 rows


,transaction_id,timestamp,card_id,merchant_id,merchant_category,amount,channel,card_present,country,international,amount_vs_card_avg,minutes_since_prev_txn,card_txns_previous_10m,is_fraud
0,TXN_0000000,2026-01-01 00:00:06+00:00,CARD_02354,MERCH_01433,restaurant,38.58,contactless,1,IN,0,0.322,NaN,0,0
1,TXN_0000001,2026-01-01 00:00:23+00:00,CARD_01829,MERCH_00331,grocery,18.91,contactless,1,IN,0,0.050,NaN,0,0
2,TXN_0000002,2026-01-01 00:01:42+00:00,CARD_00236,MERCH_01170,other,52.17,contactless,1,IN,0,0.342,NaN,0,0
3,TXN_0000003,2026-01-01 00:02:23+00:00,CARD_01088,MERCH_04901,electronics,46.65,ecommerce,0,IN,0,1.115,NaN,0,0
4,TXN_0000004,2026-01-01 00:02:29+00:00,CARD_01856,MERCH_02411,other,79.30,chip_pin,1,IN,0,0.539,NaN,0,0


In [6]:
print("checking data types of each column")
print(df.dtypes)

checking data types of each column
transaction_id                str
timestamp                     str
card_id                       str
merchant_id                   str
merchant_category             str
amount                    float64
channel                       str
card_present                int64
country                       str
international               int64
amount_vs_card_avg        float64
minutes_since_prev_txn    float64
card_txns_previous_10m      int64
is_fraud                    int64
dtype: object


In [7]:
print("checking missing values in each column")
print(df.isnull().sum())

checking missing values in each column
transaction_id               0
timestamp                    0
card_id                      0
merchant_id                  0
merchant_category            0
amount                       0
channel                      0
card_present                 0
country                      0
international                0
amount_vs_card_avg           0
minutes_since_prev_txn    2500
card_txns_previous_10m       0
is_fraud                     0
dtype: int64


In [8]:
print("check duplicate rows")
print(df.duplicated().sum())

check duplicate rows
5


In [9]:
print("analysing target column distribution")
print(df["is_fraud"].value_counts())

analysing target column distribution
is_fraud
0    98854
1     1156
Name: count, dtype: int64


In [10]:
print("analyzing targeted column distribution in percentage")
print(df["is_fraud"].value_counts(normalize=True).mul(100))

analyzing targeted column distribution in percentage
is_fraud
0    98.844116
1     1.155884
Name: proportion, dtype: float64


In [11]:
duplicated_mask = df.duplicated(keep=False)
print(duplicated_mask.sum())

10


In [12]:
df.loc[duplicated_mask,"is_fraud"].value_counts()

is_fraud
0    6
1    4
Name: count, dtype: int64

In [13]:
duplicate_groups = (
    df.groupby(df.columns.tolist(),dropna=False).size().reset_index(name="count")
)

In [14]:
duplicate_groups[duplicate_groups["count"] > 1]

,transaction_id,timestamp,card_id,merchant_id,merchant_category,amount,channel,card_present,country,international,amount_vs_card_avg,minutes_since_prev_txn,card_txns_previous_10m,is_fraud,count
5216,TXN_0005215,2026-01-05 16:16:31+00:00,CARD_01362,MERCH_03640,fuel,73.35,magstripe,1,IN,0,0.235,4862.0,0,0,2
24849,TXN_0024847,2026-01-23 08:27:59+00:00,CARD_01487,MERCH_00213,restaurant,394.60,magstripe,1,US,1,6.402,4185.0,0,1,2
25146,TXN_0025144,2026-01-23 14:48:13+00:00,CARD_01623,MERCH_04846,travel,201.84,magstripe,1,IN,0,1.554,4554.0,0,0,2
37849,TXN_0037846,2026-02-04 00:37:32+00:00,CARD_00905,MERCH_02551,electronics,58.08,ecommerce,0,IN,0,0.310,4110.0,0,1,2
87149,TXN_0087144,2026-03-20 08:48:14+00:00,CARD_01496,MERCH_04483,pharmacy,59.05,chip_pin,1,IN,0,0.614,2382.0,0,0,2


In [15]:
len(duplicate_groups)

100005

In [16]:
duplicate_groups.groupby("is_fraud")["count"].sum()

is_fraud
0    98854
1     1156
Name: count, dtype: int64

In [19]:
feature_columns = df.columns.drop("is_fraud").tolist()

identical_feature = (df.groupby(feature_columns,dropna = False).agg(
    total_rows=("is_fraud","size"),
    unique=("is_fraud","nunique")
).reset_index())
print(identical_feature.sort_values(by="total_rows",ascending=False))


      transaction_id                  timestamp     card_id  merchant_id  \
5215     TXN_0005215  2026-01-05 16:16:31+00:00  CARD_01362  MERCH_03640   
4831     TXN_0004831  2026-01-05 07:59:42+00:00  CARD_01127  MERCH_04535   
20651    TXN_0020651  2026-01-19 14:00:57+00:00  CARD_01060  MERCH_00548   
25349    TXN_0025349  2026-01-23 19:23:50+00:00  CARD_02306  MERCH_01996   
37846    TXN_0037846  2026-02-04 00:37:32+00:00  CARD_00905  MERCH_02551   
...              ...                        ...         ...          ...   
99979    TXN_0099979  2026-03-31 23:41:57+00:00  CARD_00292  MERCH_00354   
99980    TXN_0099980  2026-03-31 23:44:29+00:00  CARD_01403  MERCH_00661   
99981    TXN_0099981  2026-03-31 23:46:06+00:00  CARD_00948  MERCH_00296   
99982    TXN_0099982  2026-03-31 23:48:13+00:00  CARD_00272  MERCH_00935   
99983    TXN_0099983  2026-03-31 23:48:14+00:00  CARD_00696  MERCH_00992   

      merchant_category  amount      channel  card_present country  \
5215             